# Resampling intraday bars

Turning 5-minute OHLCV bars into 15-minute bars correctly: session filtering, label/closed conventions, dropping incomplete buckets, and checks that the result is consistent with the source.

In [ ]:
import os
from pathlib import Path

# All inputs and outputs live under DATA_DIR: data/ at the repo root by default, whether
# Jupyter was started from the repo root or from notebooks/. Set DATA_DIR to override.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = Path(os.environ.get("DATA_DIR", ROOT / "data"))
OUT_DIR = DATA_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42

In [ ]:
import numpy as np
import pandas as pd

SESSION_START, SESSION_END = "09:30", "16:00"
SOURCE_RULE, TARGET_RULE = "5min", "15min"


def synthetic_5m_bars(days=20, seed=SEED):
    rng = np.random.default_rng(seed)
    out = []
    for day in pd.bdate_range("2024-01-02", periods=days):
        idx = pd.date_range(f"{day.date()} {SESSION_START}", f"{day.date()} {SESSION_END}",
                            freq=SOURCE_RULE, inclusive="left")
        close = 100 * np.exp(np.cumsum(rng.normal(0, 0.001, len(idx))))
        open_ = np.r_[close[0], close[:-1]]
        hi = np.maximum(open_, close) * (1 + rng.uniform(0, 0.0008, len(idx)))
        lo = np.minimum(open_, close) * (1 - rng.uniform(0, 0.0008, len(idx)))
        out.append(pd.DataFrame(dict(open=open_, high=hi, low=lo, close=close,
                                     volume=rng.integers(100, 1000, len(idx))), index=idx))
    bars = pd.concat(out)
    return bars.drop(bars.sample(frac=0.01, random_state=seed).index)  # simulate a few missing bars


m5 = synthetic_5m_bars()
m5.head()

In [ ]:
def resample_ohlcv(df, rule, expected_per_bucket):
    df = df.between_time(SESSION_START, SESSION_END, inclusive="left")
    agg = {"open": "first", "high": "max", "low": "min", "close": "last", "volume": "sum"}
    out = df.resample(rule, label="left", closed="left").agg(agg)
    counts = df["close"].resample(rule, label="left", closed="left").count()
    out["n_source_bars"] = counts
    out = out[counts > 0]
    out["complete"] = out["n_source_bars"] == expected_per_bucket
    return out


m15 = resample_ohlcv(m5, TARGET_RULE, expected_per_bucket=3)
print(f"{len(m5)} 5m bars -> {len(m15)} 15m bars, {(~m15.complete).sum()} incomplete")
m15.head()

In [ ]:
# Consistency checks
assert (m15.high >= m15[["open", "close"]].max(axis=1)).all()
assert (m15.low <= m15[["open", "close"]].min(axis=1)).all()
assert m15.volume.sum() == m5.between_time(SESSION_START, SESSION_END, inclusive="left").volume.sum()
daily_close_5m = m5.close.groupby(m5.index.date).last()
daily_close_15m = m15.close.groupby(m15.index.date).last()
assert np.allclose(daily_close_5m.values, daily_close_15m.values)
print("all checks passed")